# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ismayil282/flyrank-ml-engineering-task-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**This is primarily a ranking/scoring task, built on top of a binary classifier.**

I don't just need "will this page decline: yes/no" — I need an ordered queue, because a reviewer
only has time for the top N pages, not a flat list of everyone flagged "yes." So the pipeline is:
train a classifier that outputs a probability (not just a label), then use that probability
(combined with a transparent baseline score) to rank all pages, and hand the reviewer the top of
that ranking with reason codes attached. Underneath the ranking sits a binary classification
problem — "is this page review-worthy" — but the deliverable a human actually uses is the
ranking, not the raw yes/no.

In [1]:
# No computation needed for this section — the task-type reasoning above is conceptual.


## 2. Target or proxy

**Starter proxy (what I have running now):** `is_declining_label = (trend_direction == "down")`.
This is a rule computed from the *current* 90-day window, not a future outcome — it's a
convenient beginner proxy, and I should be honest that it's a proxy, not the real target.

**Where I want to land by the model week:** a genuinely future-looking label built from the
warehouse's daily fact table, something like: using features from a page's prior 90 days, predict
whether it declines (or fails to recover) over the *next* 30 days. That moves the label from "a
bucket calculated from data I already have" to "an outcome that hasn't happened yet at
prediction time" — which is the difference between a proxy label and a real forecasting target,
and it removes any chance of the feature window leaking into the label.

Either way, the label always comes from **observed signals** (impressions, clicks, position,
sessions) — never from FlyRank's own product decision fields, since those aren't in this data and
using them would just teach a model to copy an existing rule instead of finding real signal.

In [2]:
# No computation needed for this section — see Section 4 for the label distribution.


## 3. Success metric

**Precision@50** is the metric I'll defend, not overall accuracy or even ROC AUC.

Reviewer time is the actual constraint here (see Week 1 framing), so what matters is: of the top
50 pages the ranking says to look at first, how many are genuinely worth reviewing? A model can
have a great ROC AUC and still be useless if its top-of-queue picks are wrong, because that's
exactly where a reviewer's limited attention goes. Precision@50 (or Precision@K more generally)
answers "is the front of my queue trustworthy," which is the only part of the ranking a
time-constrained human will ever actually act on. I'll track ROC AUC and average precision too,
as sanity checks that the model isn't just overfitting the top of the list, but Precision@50 is
the number that decides whether this is worth shipping.

In [3]:
# No computation needed for this section — the metric argument above is conceptual.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys, subprocess
import pandas as pd

# Works whether run in Colab, locally from work/notebooks/, or from the repo root.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ismayil282/flyrank-ml-engineering-task-1"
REPO_DIR = "flyrank-ml-engineering-task-1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # move from work/notebooks/ to the repo root

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Confirm the grain: one row per content_id (one page), not per client or per day
n_rows = len(df)
n_unique_pages = df["content_id"].nunique()
n_clients = df["client_id"].nunique()

print(f"rows: {n_rows} | unique content_id: {n_unique_pages} | clients: {n_clients}")
print("One row = one page (content_id), scoped to a client (client_id).")
print(f"Row equals unique page count: {n_rows == n_unique_pages}")

# Show what one row actually looks like, in the columns that matter for this lane
cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d", "sessions_90d",
    "avg_position", "ctr", "trend_direction", "days_since_last_update",
]
display_df = df[cols].head(5)
print()
print(display_df.to_string(index=False))

# Starter proxy label distribution (Section 2's current-window proxy)
is_declining = (df["trend_direction"] == "down")
print()
print(f"Starter proxy label rate (trend_direction == 'down'): {is_declining.mean():.1%} "
      f"({is_declining.sum()} of {n_rows} pages)")


rows: 30000 | unique content_id: 30000 | clients: 32
One row = one page (content_id), scoped to a client (client_id).
Row equals unique page count: True

          content_id         client_id  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr trend_direction  days_since_last_update
content_304f48230142 client_f369cb89fc             3803          29            17          10.6 0.76            down                      20
content_a1fb4e703a9e client_4e07408562            15320           7             9          20.3 0.05            down                      25
content_9aa793d4d895 client_7f2253d7e2            12581          11            11          36.5 0.09            down                      20
content_331d6c4de07b client_19581e27de            11751          58            78           6.2 0.49          stable                      22
content_d99b7a2d90ca client_3fdba35f04            19140          24           145          44.0 0.13            down                      14


## 5. Why ML beats a fixed rule here

A single if-statement can catch one obvious pattern ("impressions are high and position is bad")
but this problem has several weakly-correlated signals that only matter *together*: how visible a
page is, how stale it is, whether it's under-clicking for its position, and whether engagement is
weak — and the right combination differs page to page. A fixed rule has to pick fixed thresholds
and fixed weights ahead of time; a model can learn how those signals interact and weigh them
based on what actually predicted decline in the data, not on someone's guess.

The starter pipeline already shows this isn't just a hunch: on this same 30,000-row dataset with
client-holdout validation (whole clients held out of training), the hand-written baseline rule
scores 0.240 on Precision@50, while a random forest reaches 0.740 — roughly 12 correct picks in
the top 50 versus roughly 37. That's evidence a learned ranking finds real structure the fixed
rule misses, at least on this slice. My job for the rest of the internship is to check whether
that gap holds up on a stronger, future-looking label and the full warehouse — not just repeat
this number.

In [5]:
# No computation needed for this section — the baseline-vs-model numbers above are
# taken directly from outputs/model_report.md, generated by the starter pipeline
# (python scripts/run_all.py) on this same dataset.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.